# Retail Demand Forecasting - Exploratory Data Analysis

This notebook performs comprehensive EDA on the Favorita grocery sales dataset.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
from src.data.ingestion import DataIngestion
from src.data.cleaning import DataCleaner
from src.data.validation import DataValidation
from src.visualization.plots import Visualizer

plt.style.use("seaborn-v0_8")
%matplotlib inline
FIG_DIR = Path("..") / "reports" / "figures"
viz = Visualizer(save_path=str(FIG_DIR))
cleaner = DataCleaner()
validator = DataValidation()
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
ingestion = DataIngestion("../data/raw")
data = ingestion.load_data("favorita")
if not data:
    print("No data found. Attempting download...")
    success = ingestion.download_favorita()
    if success:
        data = ingestion.load_data("favorita")
    else:
        print("Please download the dataset manually. See data/raw/README.md")
        data = {}
print(f"Loaded datasets: {list(data.keys())}")

In [ ]:
df = data.get("train", pd.DataFrame())
df = cleaner.clean_column_names(df)
df = cleaner.clean_sales_data(df)
print(f"Shape: {df.shape}")
print(f"Date range: {df.date.min()} to {df.date.max()}")
print(f"\nColumns:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
print(validator.generate_validation_report(df, "train"))

In [ ]:
fig = viz.plot_sales_distribution(df, "sales")
viz.save_figure(fig, "sales_distribution.png")
plt.show()

In [ ]:
fig = viz.plot_time_series(df, "date", "sales")
viz.save_figure(fig, "time_series.png")
plt.show()

In [ ]:
fig = viz.plot_weekly_seasonality(df, "date", "sales")
viz.save_figure(fig, "weekly_seasonality.png")
plt.show()

In [ ]:
fig = viz.plot_monthly_seasonality(df, "date", "sales")
viz.save_figure(fig, "monthly_seasonality.png")
plt.show()

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
yearly_monthly = df.groupby(["year", "month"])["sales"].sum().reset_index()
fig, ax = plt.subplots(figsize=(12, 6))
for year in sorted(yearly_monthly["year"].unique()):
    year_data = yearly_monthly[yearly_monthly["year"] == year]
    ax.plot(year_data["month"], year_data["sales"], marker="o", label=str(year))
ax.set_xlabel("Month")
ax.set_ylabel("Total Sales")
ax.set_title("Year-over-Year Monthly Sales Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, "year_over_year.png")
plt.show()

In [ ]:
daily = df.groupby("date")["sales"].sum().reset_index()
daily["rolling_7"] = daily["sales"].rolling(7).mean()
daily["rolling_30"] = daily["sales"].rolling(30).mean()
daily["rolling_90"] = daily["sales"].rolling(90).mean()
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(daily["date"], daily["sales"], alpha=0.4, linewidth=0.5, label="Daily")
ax.plot(daily["date"], daily["rolling_7"], label="7-day MA", linewidth=1.5)
ax.plot(daily["date"], daily["rolling_30"], label="30-day MA", linewidth=2)
ax.plot(daily["date"], daily["rolling_90"], label="90-day MA", linewidth=2)
ax.set_xlabel("Date")
ax.set_ylabel("Total Sales")
ax.set_title("Rolling Averages")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, "rolling_averages.png")
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
daily = df.groupby("date")["sales"].sum().reset_index()
daily_sales = daily.set_index("date")["sales"]
daily_sales = daily_sales.asfreq("D").ffill()
# Use weekly period for decomposition (need 2 cycles minimum)
period = min(52, len(daily_sales) // 4)
if len(daily_sales) >= 2 * period:
    decomp = seasonal_decompose(daily_sales, model="additive", period=period)
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    decomp.observed.plot(ax=axes[0], title="Observed")
    decomp.trend.plot(ax=axes[1], title="Trend")
    decomp.seasonal.plot(ax=axes[2], title="Seasonal")
    decomp.resid.plot(ax=axes[3], title="Residual")
    plt.tight_layout()
    viz.save_figure(fig, "trend_decomposition.png")
    plt.show()
else:
    print(f"Not enough data for decomposition: need {2*period}, have {len(daily_sales)}")

In [ ]:
top_items = df.groupby("item_nbr")["sales"].sum().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(top_items)), top_items.values, color="teal", alpha=0.7)
ax.set_xticks(range(len(top_items)))
ax.set_xticklabels([str(i) for i in top_items.index], rotation=45, ha="right")
ax.set_xlabel("Item Number")
ax.set_ylabel("Total Sales")
ax.set_title("Top 20 SKUs by Total Sales")
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, "top_skus.png")
plt.show()

In [ ]:
if "store_nbr" in df.columns:
    store_sales = df.groupby("store_nbr")["sales"].sum().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(range(len(store_sales)), store_sales.values, color="steelblue", alpha=0.7)
    ax.set_xlabel("Store Number")
    ax.set_ylabel("Total Sales")
    ax.set_title("Sales by Store")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    viz.save_figure(fig, "store_analysis.png")
    plt.show()

In [ ]:
items_df = data.get("items", pd.DataFrame())
if not items_df.empty:
    items_df = cleaner.clean_column_names(items_df)
if not items_df.empty and "family" in items_df.columns:
    merged = df.merge(items_df[["item_nbr", "family"]], on="item_nbr", how="left")
    fig = viz.plot_category_analysis(merged, "family", "sales")
    viz.save_figure(fig, "category_analysis.png")
    plt.show()

In [ ]:
if "onpromotion" in df.columns:
    df["is_promotion"] = (df["onpromotion"] > 0).astype(int)
    fig = viz.plot_promotion_analysis(df, "sales")
    viz.save_figure(fig, "promotion_analysis.png")
    plt.show()
    promo_stats = df.groupby("is_promotion")["sales"].describe()
    display(promo_stats)

In [ ]:
holidays = data.get("holidays_events", pd.DataFrame())
if not holidays.empty:
    holidays = cleaner.clean_column_names(holidays)
    holidays["date"] = pd.to_datetime(holidays["date"])
    df["is_holiday"] = df["date"].isin(holidays["date"]).astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    holiday_stats = df.groupby("is_holiday")["sales"].mean()
    axes[0].bar(["Non-Holiday", "Holiday"], holiday_stats.reindex([0,1]).fillna(0).values, color=["steelblue", "coral"], alpha=0.7)
    axes[0].set_title("Average Sales: Holiday vs Non-Holiday")
    axes[0].grid(True, alpha=0.3)
    holiday_types = holidays["type"].value_counts()
    axes[1].bar(holiday_types.index, holiday_types.values, color="green", alpha=0.7)
    axes[1].set_title("Holiday Types")
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    viz.save_figure(fig, "holiday_analysis.png")
    plt.show()

In [ ]:
missing = validator.check_missing_values(df)
if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.4)))
    ax.barh(range(len(missing)), missing["missing_count"].values, color="salmon", alpha=0.7)
    ax.set_yticks(range(len(missing)))
    ax.set_yticklabels(missing.index)
    ax.set_xlabel("Missing Count")
    ax.set_title("Missing Values by Column")
    ax.invert_yaxis()
    plt.tight_layout()
    viz.save_figure(fig, "missing_values.png")
    plt.show()
else:
    print("No missing values found")

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, annot=False, cmap="RdBu", center=0, ax=ax, linewidths=0.5)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
viz.save_figure(fig, "correlation_heatmap.png")
plt.show()

In [ ]:
oil = data.get("oil", pd.DataFrame())
if not oil.empty:
    oil = cleaner.clean_column_names(oil)
    oil["date"] = pd.to_datetime(oil["date"])
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(oil["date"], oil["dcoilwtico"], color="darkgreen", linewidth=1)
    ax.set_title("Daily Oil Price")
    ax.set_xlabel("Date")
    ax.set_ylabel("Oil Price")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    viz.save_figure(fig, "oil_prices.png")
    plt.show()

In [ ]:
stores_df = data.get("stores", pd.DataFrame())
if not stores_df.empty:
    stores_df = cleaner.clean_column_names(stores_df)
    merged_stores = df.merge(stores_df, on="store_nbr", how="left")
    if "type" in merged_stores.columns:
        type_sales = merged_stores.groupby("type")["sales"].mean().sort_values()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(type_sales.index, type_sales.values, color="purple", alpha=0.7)
        ax.set_title("Average Sales by Store Type")
        ax.set_xlabel("Store Type")
        ax.set_ylabel("Average Sales")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        viz.save_figure(fig, "store_type_analysis.png")
        plt.show()

In [ ]:
print("=== EDA Summary ===")
print("Total records:", len(df))
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Avg sales:", round(df["sales"].mean(), 2))
print("Median sales:", round(df["sales"].median(), 2))
print("Sales std:", round(df["sales"].std(), 2))
print("Figures saved to reports/figures/")